## Step 1: read all sheets needed for this part

In [23]:
import re
import numpy as np
import pandas as pd
 
INPUT_PATH = "C:/Users/kar.eco/Downloads/example_output.xlsx"


# Calculations steet ---------------------------------------------------------------------------
inv_cost = pd.read_excel(INPUT_PATH, sheet_name="Inv_cost", header=None)
report_model = pd.read_excel(INPUT_PATH, sheet_name="report__model", header=None)
node_stochastic = pd.read_excel(INPUT_PATH, sheet_name="report__node__stochastic_scenar", header=None)
unit_stochastic = pd.read_excel(INPUT_PATH, sheet_name="report__unit__stochastic_scenar", header=None)
# Cost_revenue sheet ---------------------------------------------------------------------------
connection_flows = pd.read_excel(INPUT_PATH, sheet_name="report__connection__node__direc", header=None)
unit_flows = pd.read_excel(INPUT_PATH, sheet_name="report__unit__node__direction__", header=None)
prices = pd.read_excel(INPUT_PATH, sheet_name="Time_series_base", header=None)


### Calculations sheet

In [24]:
# --- Inv_cost: name / unit_cost / lifetime_years / om_rate, by row ---
inv_cost = inv_cost.dropna(subset=[0]).copy()
inv_cost.columns = list(range(inv_cost.shape[1]))  # keep plain 0-based column numbers
inv_cost["lifetime_years"] = (
    inv_cost[5].astype(str).str.extract(r"([\d.]+)").astype(float)[0] / 365
)
# columns: 0=name, 4=unit_cost (E), 7=om_rate (H), "lifetime_years" (derived from F)
inv_cost = inv_cost.set_index(inv_cost.index + 1)  # match original Excel row numbers
 
# --- number of rolling horizons = count of "total_costs" rows in report__model ---
n_horizons = (report_model[2] == "total_costs").sum()
 
# --- Storage Investment block ---
# "Total share of max" in the original sheet = SUM(...F26282:F26293), which is
# just the rows where report__node__stochastic_scenar's metric column == 'storages_invested'
storages_invested = node_stochastic[node_stochastic[2] == "storages_invested"]
node_headers = node_stochastic.iloc[0]  # e.g. "Report__ch3oh_st__realisation"
 
storage_investment = {}
for label, storage_name, unit_cost_name in [
    ("methanol", "ch3oh_st", "solar_plant"), # unit_cost taken from 'solar_plant' in the original sheet - see note below
    ("hydrogen", "h2_st", "h2_st"),
    ("power", "power_st", "power_st"),
]:
    data_col = node_headers[node_headers == f"Report__{storage_name}__realisation"].index[0]
    inv_row = inv_cost.index[inv_cost[0] == storage_name][0]
    unit_cost_row = inv_cost.index[inv_cost[0] == unit_cost_name][0]
 
    total_share_of_max = storages_invested[data_col].sum()
    total_cost = (
        total_share_of_max
        * inv_cost.loc[unit_cost_row, 4]
        * n_horizons
        * inv_cost.loc[inv_row, "lifetime_years"]
    )
    om = total_cost * inv_cost.loc[inv_row, 7]
    storage_investment[label] = {
        "total_share_of_max": total_share_of_max,
        "total_cost": total_cost,
        "O&M": om,
    }
# NOTE: the original 'methanol' formula multiplies by 'solar_plant's unit cost instead
# of 'ch3oh_st's own cost - looks like a copy-paste bug in the source file, kept here
# on purpose to match the original output. if bug change to ch3oh_st 
 
# --- Unit Investment block ---
# same idea: "Total" in the original sheet = SUM(...2:13), which is just the rows
# where report__unit__stochastic_scenar's metric column == 'units_invested'
units_invested = unit_stochastic[unit_stochastic[2] == "units_invested"]
unit_headers = unit_stochastic.iloc[0]  # e.g. "Report__electrolyzer__realisation"
 
# every unit present in report__unit__stochastic_scenar
unit_names = [
    name.replace("Report__", "").replace("__realisation", "")
    for name in unit_headers.dropna()
    if str(name).startswith("Report__")
]
# only these units had a total_cost/O&M formula in the original sheet
units_with_cost = {"co2_vaporizer", "dist_tower", "electrolyzer", "excess_heat_exchanger", "o2_liquefier", "steam_plant"}
 
unit_investment = {}
for name in unit_names:
    data_col = unit_headers[unit_headers == f"Report__{name}__realisation"].index[0]
    total = units_invested[data_col].sum()
 
    total_cost, om = np.nan, np.nan
    if name in units_with_cost:
        inv_row = inv_cost.index[inv_cost[0] == name][0]
        total_cost = total * inv_cost.loc[inv_row, 4] * n_horizons * inv_cost.loc[inv_row, "lifetime_years"]
        om = total_cost * inv_cost.loc[inv_row, 7]
    unit_investment[name] = {"total": total, "total_cost": total_cost, "O&M": om}

In [25]:
storage_investment_df = pd.DataFrame(storage_investment)
storage_investment_df

,methanol,hydrogen,power
total_share_of_max,4.462797e-01,0.0,0.0
total_cost,7.809894e+06,0.0,0.0
O&M,1.561979e+05,0.0,0.0


In [26]:
unit_investment_df = pd.DataFrame(unit_investment)
unit_investment_df

,ch3oh_reactor,co2_import,co2_vaporizer,dh_heat_exchanger,dist_tower,electrolyzer,excess_heat_exchanger,o2_liquefier,pth_dummy_unit,solar_plant,steam_plant,water_import,wind_plant
total,1.0,1.0,3.839941e-02,0.0,3.790418e-01,8.916563e-02,2.185517e-02,2.384149e-02,0.0,0.714286,2.185517e-02,1.0,0.0
total_cost,NaN,NaN,1.919970e+07,NaN,1.554071e+08,2.240287e+08,8.523515e+06,5.149762e+07,NaN,NaN,2.404068e+06,NaN,NaN
O&M,NaN,NaN,3.839941e+05,NaN,3.108143e+06,4.480573e+06,1.704703e+05,3.089857e+06,NaN,NaN,4.808137e+04,NaN,NaN


### Cost_revenue sheet

In [27]:
# report__connection__node__direc and report__unit__node__direction__ both start
# their data at row 2, same as Cost_revenue. Time_series_base has 6 extra header
# rows (Object type / Relationship class / Object name / Node / Alternative /
# Parameter name), so its data starts 6 rows lower - row 8.
connection_headers = connection_flows.iloc[0]  # e.g. "Report__pl_dh__dh__to_node__realisation"
unit_headers = unit_flows.iloc[0]              # e.g. "Report__water_import__water__to_node__realisation"
price_object_row = prices.iloc[3]              # "Object name" row
price_node_row = prices.iloc[4]                # "Node" row

date_time = connection_flows.iloc[1:, 1].reset_index(drop=True)  # column B = DateTime

# what to build for every output column:
#   flow_sheet          -> which raw sheet the flow comes from
#   flow_key            -> the object__node__direction identifier in that sheet's header
#   price_object/_node  -> identify the matching Time_series_base column by its own
#                          (Object name, Node) metadata, instead of a hardcoded letter
#   price_col_override  -> used only for 'tarif_from_dk': Time_series_base column E's
#                          "Object name"/"Node" metadata is corrupted (holds a stray
#                          number instead of text), so it can't be matched by name and
#                          is kept as a flagged positional exception
cost_revenue_spec = [
    {"label": "district_heating", "flow_sheet": "connection", "flow_key": "pl_dh__dh__to_node",
     "price_object": "pl_dh", "price_node": "dh"},
    {"label": "O2", "flow_sheet": "connection", "flow_key": "pl_o2__liquid_o2__from_node",
     "price_object": "pl_o2", "price_node": "o2_demand"},
    {"label": "wind_PPA", "flow_sheet": "connection", "flow_key": "pl_wind_PPA__power__to_node",
     "price_object": "pl_wind_PPA", "price_node": "wind_plant_node"},
    {"label": "solar_PPA", "flow_sheet": "connection", "flow_key": "pl_solar_PPA__power__to_node",
     "price_object": "pl_solar_PPA", "price_node": "solar_plant_node"},
    {"label": "power_from_de", "flow_sheet": "connection", "flow_key": "pl_wholesale_de__power__to_node",
     "price_object": "pl_wholesale_de", "price_node": "power_wholesale_de"},
    {"label": "tarif_from_de", "flow_sheet": "connection", "flow_key": "pl_wholesale_de__power_wholesale_de__from_node",
     "price_object": "pl_wholesale_de", "price_node": "power"},
    {"label": "power_from_dk", "flow_sheet": "connection", "flow_key": "pl_wholesale_dk__power__to_node",
     "price_object": "pl_wholesale_dk", "price_node": "power_wholesale_dk"},
    {"label": "tarif_from_dk", "flow_sheet": "connection", "flow_key": "pl_wholesale_dk__power__to_node",
     "price_col_override": 4},
    {"label": "water", "flow_sheet": "unit", "flow_key": "water_import__water__to_node",
     "price_object": "water_import", "price_node": "water_source"},
    {"label": "CO2", "flow_sheet": "unit", "flow_key": "co2_import__co2__to_node",
     "price_object": "co2_import", "price_node": "co2_source"},
]

cost_revenue = {}
for spec in cost_revenue_spec:
    flow_headers = connection_headers if spec["flow_sheet"] == "connection" else unit_headers
    flow_sheet_df = connection_flows if spec["flow_sheet"] == "connection" else unit_flows

    flow_col = flow_headers[flow_headers == f"Report__{spec['flow_key']}__realisation"].index[0]
    flow_values = flow_sheet_df.iloc[1:, flow_col].astype(float).reset_index(drop=True)

    if "price_col_override" in spec:
        price_col = spec["price_col_override"]
    else:
        price_col = price_object_row[
            (price_object_row == spec["price_object"]) & (price_node_row == spec["price_node"])
        ].index[0]
    price_values = prices.iloc[7:, price_col].astype(float).reset_index(drop=True)

    cost_revenue[spec["label"]] = flow_values * price_values

In [28]:
cost_revenue_df = pd.DataFrame(cost_revenue)
cost_revenue_df.insert(0, "DateTime", date_time)

print(cost_revenue_df.head())
print()
print(cost_revenue_df.drop(columns="DateTime").sum())

              DateTime  district_heating           O2      wind_PPA  \
0  2019-01-01T00:00:00       -875.647822 -6731.609967  17182.176000   
1  2019-01-01T01:00:00       -802.677170 -6731.609967  17447.014019   
2  2019-01-01T02:00:00       -729.706518 -6731.609967  17447.014019   
3  2019-01-01T03:00:00       -729.706518 -6731.609967  17447.014019   
4  2019-01-01T04:00:00       -802.677170 -6731.609967  17447.014019   

   solar_PPA  power_from_de  tarif_from_de  power_from_dk  tarif_from_dk  \
0        0.0            0.0            0.0     287.744149      51.298753   
1        0.0            0.0            0.0       0.000000       0.000000   
2        0.0            0.0            0.0       0.000000       0.000000   
3        0.0            0.0            0.0       0.000000       0.000000   
4        0.0            0.0            0.0       0.000000       0.000000   

        water        CO2  
0  572.334025  968.91493  
1  572.334025  968.91493  
2  572.334025  968.91493  
3  572.3

### Investment_Units sheet

In [29]:
# investment shares already computed in unit_investment_df (see 'calculations' step)
ELECTROLYZER_UNIT_SIZE_MW = 3000
WIND_UNIT_SIZE_MW = 3800

electrolyzer_capacity = unit_investment_df.loc["total", "electrolyzer"] * ELECTROLYZER_UNIT_SIZE_MW

wind_share = unit_investment_df.loc["total", "wind_plant"]
wind_capacity = wind_share * WIND_UNIT_SIZE_MW if wind_share > 0 else WIND_UNIT_SIZE_MW

investment_units_df = pd.DataFrame(
    {"value": [electrolyzer_capacity, wind_capacity], "unit": ["MW", "MW"]},
    index=["electrolyzer", "wind capacity"],
)

print(investment_units_df)

                   value unit
electrolyzer    267.4969   MW
wind capacity  3800.0000   MW


### NPV_factor sheet

In [32]:
lifetime = 25   # years
WACC = 0.08     # weighted average cost of capital
 
present_value_factor = (1 - (1 + WACC) ** (-lifetime)) / WACC
 
npv_factor_df = pd.DataFrame(
    {"value": [lifetime, WACC, present_value_factor]},
    index=["lifetime", "WACC", "present_value_factor"],
)
 
print(npv_factor_df)

                          value
lifetime              25.000000
WACC                   0.080000
present_value_factor  10.674776


In [37]:
# ---------------------------------------------------------------------------
# LCOM sheet: Levelized Cost of Methanol, built from the results of the
# previous steps: storage_investment_df, unit_investment_df, cost_revenue_df,
# present_value_factor
# ---------------------------------------------------------------------------

# --- Investment ---
storage_investment_total = storage_investment_df.loc["total_cost"].sum()
unit_investment_total = unit_investment_df.loc["total_cost"].sum()  # unit_investment_df is unit-as-columns (Option B)
connection_investment_total = 0  # no connections have investment cost in this model

total_investment = storage_investment_total + unit_investment_total + connection_investment_total

# --- Annual cost / revenue ---
# NOTE on sign convention: Cost_revenue's revenue columns (district_heating, O2) come
# out NEGATIVE by construction (selling energy is a negative cost in the underlying
# optimization objective), so "variable_cost + total_revenue" already nets revenue out
# even though it's a plain sum, not a subtraction.
variable_cost_columns = ["wind_PPA", "solar_PPA", "power_from_de", "tarif_from_de",
                          "power_from_dk", "tarif_from_dk", "water", "CO2"]
revenue_columns = ["district_heating", "O2"]

variable_cost = cost_revenue_df[variable_cost_columns].sum().sum()
total_revenue = cost_revenue_df[revenue_columns].sum().sum()

# NOTE on a likely bug in the original workbook, preserved here on purpose: the
# original O&M cost formula was `SUM(calculations!F10:O10) + SUM(calculations!F9:O9)`.
# Row 10 of 'calculations' doesn't exist (the sheet only goes to row 9), so that first
# SUM silently contributes 0 - meaning storage O&M is never actually included, only
# unit O&M is. We reproduce that (storage_om is computed but NOT added below) to match
# the original output; storage_om is still shown so the gap is visible if you want to fix it.
storage_om = storage_investment_df.loc["O&M"].sum()   # computed, but NOT included below (matches original bug)
unit_om = unit_investment_df.loc["O&M"].sum()
om_cost = unit_om  # + storage_om is what the original formula *should* have done

total_annual_cost = om_cost + variable_cost

# --- Production ---
# total_production MWh = SUM(report__connection__node__direc!D2:D8761), i.e. the
# methanol flow delivered to demand (pl_ch3oh_demand__ch3oh__from_node)
connection_flows = pd.read_excel(INPUT_PATH, sheet_name="report__connection__node__direc", header=None)
connection_headers = connection_flows.iloc[0]
production_col = connection_headers[
    connection_headers == "Report__pl_ch3oh_demand__ch3oh__from_node__realisation"
].index[0]
total_production_mwh = connection_flows.iloc[1:, production_col].astype(float).sum()
total_production_t = total_production_mwh * 3.6 / 19.9  # MWh -> GJ (x3.6) -> t (LHV 19.9 GJ/t methanol)

# --- Levelized cost ---
lcom_per_mwh = (
    (total_investment + (variable_cost + total_revenue + om_cost) * present_value_factor)
    / (total_production_mwh * present_value_factor)
)
lcom_per_t = lcom_per_mwh / 3.6 * 19.9

lcom_df = pd.DataFrame(
    {
        "value": [
            storage_investment_total, unit_investment_total, connection_investment_total, total_investment,
            variable_cost, om_cost, total_annual_cost, total_revenue,
            total_production_mwh, total_production_t,
            lcom_per_mwh, lcom_per_t,
        ]
    },
    index=[
        "Storage Investment", "Unit Investment", "Connection Investment", "Total_Investment",
        "variable cost", "O&M cost", "total annual cost", "Total_revenue",
        "total_production MWh", "total_production t",
        "LCOM/MWh", "LCOM/t",
    ],
)

print(lcom_df)

                              value
Storage Investment     7.809894e+06
Unit Investment        4.610607e+08
Connection Investment  0.000000e+00
Total_Investment       4.688706e+08
variable cost          1.070687e+08
O&M cost               1.128112e+07
total annual cost      1.183498e+08
Total_revenue         -3.928044e+07
total_production MWh   7.321300e+05
total_production t     1.324456e+05
LCOM/MWh               1.679928e+02
LCOM/t                 9.286268e+02


In [38]:
with pd.ExcelWriter('C:/Users/kar.eco/Downloads/test.xlsx', engine='openpyxl') as writer:
    unit_investment_df.to_excel(writer, sheet_name='unit_investment_df', index=True)
    storage_investment_df.to_excel(writer, sheet_name='storage_investment_df', index=True)
    cost_revenue_df.to_excel(writer, sheet_name='cost_revenue_df', index=False)
    investment_units_df.to_excel(writer, sheet_name='investment_units_df', index=True)
    npv_factor_df.to_excel(writer, sheet_name='npv_factor_df', index=True)
    lcom_df.to_excel(writer, sheet_name='lcom_df', index=True)